# 🎓 Syllabus Design Agent
> **OBE · NBA/NAAC · Revised Bloom's Taxonomy · SDG-Aligned**  
> Powered by **Google Gemini 2.0 Flash**

---
### How to use this notebook
1. Run **Cell 1** — installs all dependencies  
2. Run **Cell 2** — paste your Gemini API key  
3. Run **Cell 3** — fill in your course details  
4. Run **Cell 4** — generates the full 15-section syllabus + PDF  
5. Run **Cell 5** — downloads the PDF to your computer  

Get a free Gemini API key at: https://aistudio.google.com/app/apikey

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────
!pip install google-genai reportlab python-dotenv rich --quiet
print('✅ Dependencies installed')

In [ ]:
# ── CELL 2: Clone repo & set Gemini API key ───────────────────────
import os, sys

# Clone the repo into Colab runtime
if not os.path.exists('/content/syllabus-design-agent'):
    !git clone https://github.com/mitaoehodaimlai-create/syllabus-design-agent.git /content/syllabus-design-agent --quiet
    print('✅ Repo cloned')
else:
    print('✅ Repo already present')

# Add to Python path
sys.path.insert(0, '/content/syllabus-design-agent')

# Set your Gemini API key
from getpass import getpass
GEMINI_API_KEY = getpass('🔑 Paste your Gemini API key: ')
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print('✅ API key set')

In [ ]:
# ── CELL 3: Configure your course ────────────────────────────────
# Edit the values below for your course

from models import CourseInput

course = CourseInput(
    course_title          = 'Machine Learning',       # ← Change this
    course_code           = 'CS-601',
    class_year            = 'TY',                     # FY / SY / TY / B.Tech / M.Tech
    department            = 'Computer Engineering',
    program               = 'B.Tech',
    discipline            = 'Computer Engineering',
    university            = 'MIT Academy of Engineering, Pune',
    course_type           = 'Hybrid',                 # Theory / Lab / Hybrid
    lectures_per_week     = 3,
    tutorials_per_week    = 1,
    practicals_per_week   = 2,
    total_credits         = 5,
    total_contact_hours   = 60,
    num_units             = 5,
    semester_duration_weeks = 16,
    prerequisites         = 'Linear Algebra, Python Programming',
    num_lab_modules       = 5,
    num_lab_assignments   = 10,
    pso_count             = 3,
)

print('✅ Course configured:')
print(course.summary())

In [ ]:
# ── CELL 4: Generate full syllabus (all 15 sections) ─────────────
import os
from pathlib import Path
from agent import SyllabusAgent, save_full_markdown, _safe_filename
from pdf_generator import generate_pdf

OUTPUT_DIR = '/content/syllabus_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('🚀 Starting syllabus generation — this takes 3–5 minutes...\n')
print('Each section streams live below as it is generated.\n')
print('─' * 60)

agent    = SyllabusAgent(api_key=os.environ['GEMINI_API_KEY'])
sections = agent.generate(course, output_dir=OUTPUT_DIR)

# Save full markdown
md_path  = save_full_markdown(sections, course, OUTPUT_DIR)
print(f'\n✅ Markdown saved: {md_path}')

# Generate PDF
slug     = _safe_filename(course.course_title)
pdf_path = Path(OUTPUT_DIR) / f'{slug}_FULL_SYLLABUS.pdf'
meta = {
    'Program'     : f'{course.program} — {course.discipline}',
    'Department'  : course.department,
    'Course Code' : course.course_code,
    'Credits'     : str(course.total_credits),
    'L-T-P'       : course.ltp,
    'University'  : course.university,
}
generate_pdf(md_path.read_text(), pdf_path, course_title=course.course_title, meta=meta)
print(f'✅ PDF saved:      {pdf_path}')
print(f'\n📄 PDF size: {pdf_path.stat().st_size:,} bytes')
print('\n🎉 Generation complete! Run Cell 5 to download the PDF.')

In [ ]:
# ── CELL 5: Download the PDF ──────────────────────────────────────
from google.colab import files
files.download(str(pdf_path))
print(f'📥 Downloading: {pdf_path.name}')

In [ ]:
# ── CELL 6 (optional): Download the full Markdown too ────────────
from google.colab import files
files.download(str(md_path))
print(f'📥 Downloading: {md_path.name}')

In [ ]:
# ── CELL 7 (optional): Regenerate one section only ───────────────
# Change section_key to one of:
#   core | lesson_plan | lab_strategy | assignments | resources

section_key = 'lesson_plan'   # ← Change this

from agent import _extract_co_unit_context
core_md = Path(OUTPUT_DIR) / f'{slug}_core.md'
co_ctx  = _extract_co_unit_context(core_md.read_text()) if core_md.exists() else ''

print(f'🔄 Regenerating section: {section_key}\n')
new_text = agent.regenerate_section(section_key, course, co_ctx)
(Path(OUTPUT_DIR) / f'{slug}_{section_key}.md').write_text(new_text)
print(f'\n✅ Section regenerated and saved.')